# 03: Try the pork-rasher defect detector

This runs the **final trained model** (YOLOv8n, 640 px) on tray images and draws what it finds. There's nothing to install, and no GPU or dataset is needed.

**How to use:** click **Runtime → Run all**. It takes about a minute: the first cell downloads the code and model.

- **Coloured boxes** are the model's detections, with their confidence.
- **White dashed boxes** are what the human labellers marked, shown for the example images.
- **The banner** shows the frame verdict: red **DEFECT** if any detection reaches the threshold, green **NO DEFECT VISIBLE** otherwise.

In [ ]:
# Get the code and the trained model (skipped if you're already inside the repository)
import os, sys, subprocess
from pathlib import Path

if not Path('src/porkinspect').exists():
    if not Path('36137316-pork-rasher-inspection').exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/mmal0078/36137316-pork-rasher-inspection'], check=True)
    os.chdir('36137316-pork-rasher-inspection')
try:
    import ultralytics  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
sys.path.insert(0, 'src')
print('Ready in', os.getcwd())

## Settings

**Threshold:** how confident the model must be before a frame is flagged. **0.16** was chosen on validation images so that at least 95% of defect frames are caught.
- Lower it to catch more defects, with more false alarms.
- Raise it to get fewer false alarms, but more misses.

In [ ]:
THRESHOLD = 0.16  # @param {type:"slider", min:0.01, max:0.95, step:0.01}

import csv
import io
from IPython.display import display, Image as IPyImage
from ultralytics import YOLO
from porkinspect.inspection import inspect_image, draw_report

model = YOLO('models/yolov8n_640_best.pt')
examples = {r['file']: r for r in csv.DictReader(open('examples/examples.csv'))}


def show(path, title=''):
    """Run the detector on one image, print the result and display the drawn image."""
    path = Path(path)
    try:
        rep = inspect_image(path, None, threshold=THRESHOLD, model=model)
    except ValueError as e:  # missing or corrupt file: reported, never treated as 'no defect'
        print(f'{path.name}: cannot inspect: {e}')
        return
    truth = []
    ex = examples.get(path.name)
    if ex and ex['labelled_boxes_xywhn']:
        for item in ex['labelled_boxes_xywhn'].split(';'):
            name, nums = item.split(':')
            truth.append((name, tuple(float(v) for v in nums.split(','))))
    print(f"--- {title or path.name}")
    if ex:
        print(f"    labelled: {ex['labelled_defects']}   |   test-set outcome: {ex['outcome_on_test_set']}")
    found = ', '.join(f"{d['class']} ({d['confidence']:.2f})" for d in rep['detections']) or 'nothing above threshold'
    print(f"    model:    {'DEFECT' if rep['defect_present'] else 'no defect visible'}   |   found: {found}"
          f"   |   {rep['inference_ms']:.0f} ms")
    for w in rep['warnings']:
        print('    warning:', w)
    img = draw_report(path, rep, truth=truth, threshold=THRESHOLD)
    img.thumbnail((720, 540))
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)  # JPEG keeps the notebook small
    display(IPyImage(data=buf.getvalue(), format='jpeg'))

## 1. Example images from the test set

These 8 images were never used in training.
- **4 successes:** each defect type found with the right type, and a clean frame correctly passed.
- **4 failures:** a defect flagged as the wrong type, a frame where only one of two defects is found, a missed defect, and a false alarm.

Each one prints what was labelled and what the model did.

In [ ]:
for f in sorted(examples):
    show(Path('examples') / f)

## 2. Try your own image

In Colab, running the next cell opens an upload button. Choose one or more JPEG or PNG images.

The model was trained on **one camera view**: an overhead view of clear trays on a dark mesh belt. Images from other setups will still give an answer, but it can't be trusted. The input checks warn about unusual sizes or brightness, but they can't detect a different scene.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        show(name)
except ImportError:
    print('Not running in Colab: call show("path/to/image.jpg") instead.')

## Limitations

- **Frame-level only:** a frame usually shows part of a tray, so *no defect visible* does not mean *tray OK*.
- **Test results** (80 images, threshold 0.16): 64 of 68 defect frames caught and 3 of 12 defect-free frames flagged.
  - Twisted-meat and loose-meat had very few training examples (about 23 each), so they are the least reliable.
- **Model licence:** AGPL-3.0 (derived from Ultralytics YOLOv8).
- **Example images:** CC BY 4.0, from Roboflow Universe ([dataset](https://universe.roboflow.com/hello-2aqe0/pork-rasher-error-packaging/dataset/4)).
- **Full details:** see the repository README.